In [16]:
#Web Access Checking
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
}
# Send a GET request to the specified URL
response = requests.get('https://www.kompas.com/')

# Check if the request was successful (status code 200)
if response.status_code == 200:
  html_content = response.content
  print(html_content)
else:
	print(response)

b'<!DOCTYPE html>\n<html>\n\n<head>\n    <script>\n        window.dataLayer = window.dataLayer || [];\n        window.dataLayer.push({\n            "title": "Berita Terkini Hari Ini, Kabar Akurat Terpercaya - Kompas.com",\n            "description": "Kompas.com - Berita Indonesia dan Dunia Terkini Hari Ini, Kabar Harian Terbaru Terpercaya Terlengkap Seputar Politik, Ekonomi, Travel, Teknologi, Otomotif, Bola",\n            "keywords": "Berita Terkini, Berita Hari Ini, Berita Harian, Berita Terbaru, Berita Akurat, Berita Terpercaya, Berita indonesia, Berita Terpopuler, Berita, Info Terkini, Jernih Melihat Dunia, Kompas",\n            "content_category": "home",\n            "canonical": "https://www.kompas.com",\n            "subscription": "False"\n        });\n    </script>\n\n    \n   \n<!-- Google Tag Manager -->\n<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({\'gtm.start\':\nnew Date().getTime(),event:\'gtm.js\'});var f=d.getElementsByTagName(s)[0],\nj=d.createElement(s),dl=

In [18]:
# Parse HTML

soup = BeautifulSoup(response.text, 'html.parser')

# HTML Structure

print(soup.prettify())

<!DOCTYPE html>
<html>
 <head>
  <script>
   window.dataLayer = window.dataLayer || [];
        window.dataLayer.push({
            "title": "Berita Terkini Hari Ini, Kabar Akurat Terpercaya - Kompas.com",
            "description": "Kompas.com - Berita Indonesia dan Dunia Terkini Hari Ini, Kabar Harian Terbaru Terpercaya Terlengkap Seputar Politik, Ekonomi, Travel, Teknologi, Otomotif, Bola",
            "keywords": "Berita Terkini, Berita Hari Ini, Berita Harian, Berita Terbaru, Berita Akurat, Berita Terpercaya, Berita indonesia, Berita Terpopuler, Berita, Info Terkini, Jernih Melihat Dunia, Kompas",
            "content_category": "home",
            "canonical": "https://www.kompas.com",
            "subscription": "False"
        });
  </script>
  <!-- Google Tag Manager -->
  <script>
   (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true

### CNBC 2019-2025

In [40]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime, timedelta

# --- KONFIGURASI ---
KEYWORDS = ["BBCA", "Bank Central Asia", "BCA", "Saham BBCA", "Saham BCA"]
START_DATE = "2019-01-01"
END_DATE = "2025-10-03"  # contoh 10 hari dulu biar cepat
OUTPUT_FILE = "scraping_cnbc_BBCA_perhari.csv"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_article_details(article_url):
    try:
        time.sleep(1)
        page = requests.get(article_url, headers=HEADERS, timeout=20)
        if page.status_code != 200:
            return None

        soup = BeautifulSoup(page.text, 'html.parser')
        title = 'Title Not Found'
        publish_date = 'Date Not Found'

        # ambil dari script dataLayer
        scripts = soup.find_all('script', type='text/javascript')
        for script in scripts:
            if script.string and 'dataLayer' in script.string:
                title_match = re.search(r"'originalTitle'\s*:\s*'([^']*)'", script.string)
                if title_match:
                    title = title_match.group(1)

                date_match = re.search(r"'publishDate'\s*:\s*'([^']*)'", script.string)
                if date_match:
                    publish_date = date_match.group(1)
                break

        return {
            'Publish_date': publish_date,
            'URL': article_url,
            'Title': title
        }

    except Exception:
        return None


def day_range(start, end):
    """Generate list of tanggal harian"""
    current = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    result = []
    while current <= end:
        date_str = current.strftime("%Y/%m/%d")
        result.append((date_str, date_str))
        current += timedelta(days=1)
    return result


if __name__ == "__main__":
    list_artikel = []

    for keyword in KEYWORDS:
        print(f"\n🔍 Keyword: {keyword}")
        for start_d, end_d in day_range(START_DATE, END_DATE):
            print(f"\n📅 Hari: {start_d}")

            # ✅ Berhenti jika sudah sampai tanggal akhir
            if start_d == END_DATE:
                print("⏹️ Sudah mencapai tanggal akhir. Scraping dihentikan.")
                break

            search_url = f"https://www.cnbcindonesia.com/search?query={keyword}&fromdate={start_d}&todate={end_d}&page=1"
            hari_data = []  # kumpulan data untuk tanggal ini

            try:
                main_page = requests.get(search_url, headers=HEADERS, timeout=15)
                if main_page.status_code != 200:
                    continue
                soup = BeautifulSoup(main_page.text, 'html.parser')
                article_links = soup.select('article a.group')
                if not article_links:
                    print("❌ Tidak ada artikel ditemukan hari ini.")
                    continue

                for link_tag in article_links:
                    if link_tag.has_attr('href'):
                        article_url = link_tag['href']
                        article_data = get_article_details(article_url)
                        if article_data:
                            article_data['Keyword'] = keyword
                            article_data['Date'] = start_d
                            hari_data.append(article_data)
                            list_artikel.append(article_data)

                # tampilkan ringkasan hari ini
                df_hari = pd.DataFrame(hari_data)
                print("✅ Jumlah artikel hari ini:", len(df_hari))
                if not df_hari.empty:
                    print(df_hari[['Publish_date', 'Title']].head())

                time.sleep(2)

            except Exception as e:
                print("⚠️", e)

    # ✅ Simpan hasil scraping setelah semua selesai
    if list_artikel:
        df = pd.DataFrame(list_artikel)
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n💾 Data berhasil disimpan ke {OUTPUT_FILE}")
    else:
        print("\n⚠️ Tidak ada data yang berhasil diambil.")



🔍 Keyword: BBCA

📅 Hari: 2019/01/01
✅ Jumlah artikel hari ini: 1
          Publish_date                                              Title
0  2019/01/01 12:25:44  Ini 5 Saham yang Paling Banyak Diborong Asing ...

📅 Hari: 2019/01/02
✅ Jumlah artikel hari ini: 2
          Publish_date                                              Title
0  2019/01/02 14:40:22  Akuisisi Dua Bank, BCA Tak Perlu Merger Anak U...
1  2019/01/02 12:39:45  Asing Borong Saham Big Cap saat IHSG di Zona M...

📅 Hari: 2019/01/03
✅ Jumlah artikel hari ini: 3
          Publish_date                                              Title
0  2019/01/03 21:45:15  Kemenkeu Tawarkan Surat Utang Ketengan Mulai 1...
1  2019/01/03 08:51:44  Simak Racikan Jitu Broker untuk Perdagangan Ha...
2  2019/01/03 08:42:23  Berbagai Rencana Akuisisi Awali Aksi Emiten di...

📅 Hari: 2019/01/04
✅ Jumlah artikel hari ini: 1
          Publish_date                                              Title
0  2019/01/04 14:16:30  Setahun Berlalu, Invest

KeyboardInterrupt: 

In [43]:
df = pd.DataFrame(list_artikel)

# ubah kolom tanggal ke format datetime dulu biar bisa diurutkan
df['Publish_date'] = pd.to_datetime(df['Publish_date'], errors='coerce')

# urutkan dari tanggal paling kecil (terlama)
df = df.sort_values(by='Publish_date', ascending=True)

# simpan ulang ke CSV
df.to_csv("scraping_cnbc_BBCA_sorted.csv", index=False, encoding='utf-8-sig')

print("✅ Data berhasil diurutkan dan disimpan ke scraping_cnbc_BBCA_sorted.csv")


✅ Data berhasil diurutkan dan disimpan ke scraping_cnbc_BBCA_sorted.csv


### 2015 Detik Finance

In [6]:
import requests as req
from bs4 import BeautifulSoup as bs
import csv
import datetime
import time
from typing import List, Dict
import os
import re
from tqdm import tqdm
from urllib.parse import quote

# KEYWORDS untuk filter artikel
KEYWORDS = ["BBCA", "Bank Central Asia", "BCA"]

def save_debug_html(html_content: str, filename: str):
    """Simpan HTML untuk debugging"""
    debug_dir = "debug_html"
    if not os.path.exists(debug_dir):
        os.makedirs(debug_dir)
    
    filepath = os.path.join(debug_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return filepath


def scrape_article_content(url: str, headers: dict, debug: bool = False) -> str:
    """Scrape konten artikel dari URL Detik"""
    try:
        time.sleep(1.5)
        res = req.get(url, timeout=25, headers=headers)
        if res.status_code != 200:
            if debug:
                print(f"         ❌ HTTP Status: {res.status_code}")
            return ""
        
        soup = bs(res.text, 'lxml')

        # Debug mode: simpan HTML
        if debug:
            filename = re.sub(r'[^\w\-_]', '_', url.split('/')[-1][:50]) + "_article.html"
            saved_path = save_debug_html(res.text, filename)
            print(f"         🐞 Debug HTML: {saved_path}")
        
        # Daftar kemungkinan container konten
        selectors = [
            ('div', 'detail__body-text'),
            ('div', 'itp_bodycontent'),
            ('div', 'detail-content'),
            ('div', 'itp_bodycontent_wrapper'),
            ('div', 'text_detail'),
            ('div', 'detail_text'),
            ('div', 'text-detail'),
            ('div', 'isi_artikel'),
            ('div', 'detail__body'),
            ('div', 'detail_text')
        ]
        
        content_div = None
        for tag, class_name in selectors:
            content_div = soup.find(tag, class_=class_name)
            if content_div:
                if debug:
                    print(f"         🎯 Konten ditemukan: <{tag} class='{class_name}'>")
                break
        
        # Ambil teks
        paragraphs = []
        exclude_prefixes = ['Baca juga', 'Simak', 'ADVERTISEMENT', 'Lihat juga', 'Saksikan']
        
        if content_div:
            for p in content_div.find_all('p'):
                text = p.get_text(strip=True)
                if not text:
                    continue
                if any(text.startswith(prefix) for prefix in exclude_prefixes):
                    continue
                paragraphs.append(text)

        # Fallback jika paragraf kosong
        if not paragraphs:
            if debug:
                print("         ⚠️  Fallback: ambil semua <p> di halaman...")
            for p in soup.find_all('p'):
                text = p.get_text(strip=True)
                if len(text) > 30 and not any(text.startswith(prefix) for prefix in exclude_prefixes):
                    paragraphs.append(text)
        
        # Gabung hasil
        content = ' '.join(paragraphs).strip()

        # Jika masih kosong, coba semua <div> berisi kalimat panjang
        if not content or len(content) < 100:
            long_divs = [div.get_text(strip=True) for div in soup.find_all('div') if len(div.get_text(strip=True)) > 100]
            if long_divs:
                content = ' '.join(long_divs[:3])
                if debug:
                    print("         🧩 Mengambil konten alternatif dari <div> panjang")

        # Simpan meskipun pendek
        if len(content) < 100:
            if debug:
                print(f"         ⚠️  Konten pendek ({len(content)} karakter) — tetap disimpan.")
        else:
            if debug:
                print(f"         ✅ Konten panjang ({len(content)} karakter)")

        return content

    except req.exceptions.Timeout:
        if debug:
            print("         ⚠️  Timeout saat mengakses artikel")
        return ""
    except Exception as e:
        if debug:
            print(f"         ⚠️  Error ambil konten: {type(e).__name__} - {e}")
        return ""


def extract_date_from_text(date_text: str) -> str:
    """Ekstrak dan format tanggal dari teks Detik"""
    try:
        if ',' in date_text:
            date_part = date_text.split(',')[1].strip()
            date_only = ' '.join(date_part.split()[:3])
            months = {
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
                'Mei': '05', 'Jun': '06', 'Jul': '07', 'Agu': '08',
                'Sep': '09', 'Okt': '10', 'Nov': '11', 'Des': '12',
                'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
                'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
                'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
            }
            parts = date_only.split()
            if len(parts) == 3:
                day, month, year = parts
                month_num = months.get(month, month)
                return f"{year}-{month_num}-{day.zfill(2)}"
    except:
        pass
    return date_text


def scrape_search_results(keyword: str, page: int, headers: dict, debug: bool = False,
                         start_date: str = None, end_date: str = None) -> List[Dict]:
    """Scrape hasil pencarian dari Detik.com"""
    articles = []
    search_url = f"https://www.detik.com/search/searchall?query={quote(keyword)}&page={page}&sortby=time"
    
    if start_date and end_date:
        try:
            start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
            end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d")
            fromdatex = start_dt.strftime("%d/%m/%Y")
            todatex = end_dt.strftime("%d/%m/%Y")
            search_url += f"&fromdatex={fromdatex}&todatex={todatex}"
            if debug:
                print(f"   📅 Filter tanggal: {fromdatex} - {todatex}")
        except:
            pass
    
    try:
        print(f"   🔍 Mengakses: {search_url}")
        time.sleep(2)
        res = req.get(search_url, timeout=25, headers=headers)
        
        if res.status_code != 200:
            print(f"   ❌ HTTP {res.status_code}")
            return articles
        
        soup = bs(res.text, 'lxml')
        if debug:
            safe_keyword = re.sub(r'[^\w\-_]', '_', keyword)
            debug_path = save_debug_html(res.text, f"search_{safe_keyword}_page{page}.html")
            print(f"   🐞 Debug HTML: {debug_path}")
        
        article_items = soup.find_all('article') or soup.find_all('div', class_='list-content__item')
        
        if debug:
            print(f"   🎯 Ditemukan {len(article_items)} artikel")
        if not article_items:
            print(f"   ⚠️  Tidak ada artikel ditemukan di halaman ini")
            return articles
        
        for item in article_items:
            try:
                title = None
                link = None
                released = ""
                title_tag = item.find('h3', class_='media__title') or \
                            item.find('h2', class_='media__title') or \
                            item.find('a', class_='media__link')
                if title_tag:
                    if title_tag.name == 'a':
                        title = title_tag.text.strip()
                        link = title_tag.get('href')
                    else:
                        a_tag = title_tag.find('a')
                        if a_tag:
                            title = a_tag.text.strip()
                            link = a_tag.get('href')
                
                date_tag = item.find('div', class_='media__date') or \
                           item.find('span', class_='media__date')
                if date_tag:
                    released = extract_date_from_text(date_tag.text.strip())
                
                if not title or not link:
                    continue
                
                if not link.startswith('http'):
                    link = 'https://www.detik.com' + link
                
                if 'detik.com' not in link:
                    continue
                
                articles.append({'title': title, 'released': released, 'url': link})
            except Exception as e:
                if debug:
                    print(f"   ⚠️  Error parsing item: {type(e).__name__} - {e}")
                continue
        
        return articles
    except req.exceptions.Timeout:
        print(f"   ❌ Timeout saat mengakses halaman pencarian")
        return articles
    except Exception as e:
        print(f"   ❌ Error scraping halaman: {type(e).__name__} - {e}")
        return articles


# Bagian utama tetap sama (tidak diubah)
# Jadi kamu bisa lanjut dari fungsi `sc_detik_search()` di bawah
# salin kode kamu mulai dari def sc_detik_search(...) sampai akhir



def sc_detik_search(keywords: List[str],
                    max_pages: int = None,
                    output_file: str = 'ress_detik.csv',
                    debug: bool = False,
                    start_date: str = None,
                    end_date: str = None):
    """
    Scraping Detik Finance berdasarkan keyword.
    Jika max_pages=None, maka scraping akan berjalan sampai tidak ada artikel baru.
    """
    
    print(f"\n{'='*70}")
    print(f"🔍 Keywords: {', '.join(keywords)}")
    print(f"📄 Mode halaman: {'SEMUA' if max_pages is None else max_pages}")
    if start_date and end_date:
        print(f"📅 Rentang tanggal: {start_date} s/d {end_date}")
    print(f"💾 Output: {output_file}")
    if debug:
        print(f"🐞 Debug mode aktif")
    print(f"{'='*70}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
    }

    all_articles = []
    seen_urls = set()

    # === SCRAPE SEARCH RESULTS ===
    for kw_idx, keyword in enumerate(keywords, 1):
        print(f"\n[{kw_idx}/{len(keywords)}] 🔑 Keyword: '{keyword}'")
        print(f"{'-'*70}")

        page = 1
        keyword_articles = []

        while True:
            print(f"   📄 Halaman {page}")
            articles = scrape_search_results(keyword, page, headers, debug=debug,
                                             start_date=start_date, end_date=end_date)

            if not articles:
                print(f"   ⚠️  Tidak ada artikel ditemukan, berhenti.")
                break

            new_articles = [a for a in articles if a['url'] not in seen_urls]
            for art in new_articles:
                seen_urls.add(art['url'])
            keyword_articles.extend(new_articles)

            print(f"   ➕ Artikel baru: {len(new_articles)}")

            # Hentikan kondisi
            if not new_articles or len(articles) < 5:
                break
            if max_pages is not None and page >= max_pages:
                print(f"   ⛔ Batas halaman {max_pages} tercapai.")
                break

            page += 1
            time.sleep(2)

        print(f"✅ Total artikel keyword '{keyword}': {len(keyword_articles)}")
        all_articles.extend(keyword_articles)
        time.sleep(3)

    if not all_articles:
        print("\n❌ Tidak ada artikel yang ditemukan untuk semua keyword.")
        return

    # === SCRAPE CONTENT ===
    print(f"\n{'='*70}")
    print("📥 MENGAMBIL KONTEN ARTIKEL")
    print(f"{'='*70}\n")

    scraped_data = []
    success_count = 0
    failed_count = 0

    for art in tqdm(all_articles, desc="Scraping artikel", ncols=70):
        content = scrape_article_content(art['url'], headers, debug=debug)
        if not content or len(content) < 100:
            failed_count += 1
            continue

        scraped_data.append({
            'title': art['title'],
            'released': art['released'],
            'url': art['url'],
            'content': content
        })
        success_count += 1
        time.sleep(1)

    # === SAVE TO CSV ===
    print(f"\n{'='*70}")
    print(f"💾 Menyimpan hasil ke {output_file}...")
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['title', 'released', 'url', 'content'], quoting=csv.QUOTE_ALL)
            writer.writeheader()
            writer.writerows(scraped_data)
        print("✅ File tersimpan!\n")
    except Exception as e:
        print(f"❌ Error saat menyimpan file: {e}")

    # === SUMMARY ===
    print(f"{'='*70}")
    print("📊 HASIL AKHIR")
    print(f"{'='*70}")
    print(f"Total artikel ditemukan: {len(all_articles)}")
    print(f"Berhasil di-scrape: {success_count}")
    print(f"Gagal di-scrape: {failed_count}")
    print(f"File output: {output_file}")
    print(f"{'='*70}")
    print("✅ SCRAPING SELESAI!\n")


# === MAIN ===
if __name__ == '__main__':
    print("="*70)
    print("📰 DETIK FINANCE SCRAPER")
    print("="*70)

    default_keywords = ["BBCA", "BCA", "Bank Central Asia"]
    print(f"\n🔍 Keywords default: {', '.join(default_keywords)}")
    use_default = input("Gunakan default keywords? (y/n): ").strip().lower() != 'n'
    keywords = default_keywords if use_default else [
        k.strip() for k in input("Masukkan keyword (pisahkan dengan koma): ").split(',') if k.strip()
    ]

    max_pages_input = input("\n📄 Max halaman per keyword (Enter = semua): ").strip()
    max_pages = int(max_pages_input) if max_pages_input.isdigit() else None

    date_filter = input("\n📅 Aktifkan filter tanggal? (y/n): ").strip().lower() == 'y'
    start_date = end_date = None
    if date_filter:
        start_date = input("   Tanggal MULAI (YYYY-MM-DD): ").strip()
        end_date = input("   Tanggal SELESAI (YYYY-MM-DD): ").strip()

    output_file = input("\n💾 Nama file output (Enter = ress_detik.csv): ").strip() or 'ress_detik.csv'
    debug = input("\n🐞 Aktifkan DEBUG MODE? (y/n): ").strip().lower() == 'y'

    print(f"\n{'='*70}")
    print("📋 KONFIRMASI")
    print(f"{'='*70}")
    print(f"Keywords: {', '.join(keywords)}")
    print(f"Max halaman: {max_pages or 'SEMUA'}")
    if date_filter:
        print(f"Filter tanggal: {start_date} s/d {end_date}")
    print(f"Output file: {output_file}")
    print(f"Debug: {'ON' if debug else 'OFF'}")
    print(f"{'='*70}")

    if input("\n▶️  Lanjutkan scraping? (y/n): ").strip().lower() == 'y':
        print("\n🚀 Mulai scraping...\n")
        sc_detik_search(keywords, max_pages, output_file, debug, start_date, end_date)
    else:
        print("\n❌ Dibatalkan oleh pengguna.")

📰 DETIK FINANCE SCRAPER

🔍 Keywords default: BBCA, BCA, Bank Central Asia


Gunakan default keywords? (y/n):  y

📄 Max halaman per keyword (Enter = semua):  semua

📅 Aktifkan filter tanggal? (y/n):  y
   Tanggal MULAI (YYYY-MM-DD):  2015-01-01
   Tanggal SELESAI (YYYY-MM-DD):  2015-12-31

💾 Nama file output (Enter = ress_detik.csv):  detik2015.csv

🐞 Aktifkan DEBUG MODE? (y/n):  y



📋 KONFIRMASI
Keywords: BBCA, BCA, Bank Central Asia
Max halaman: SEMUA
Filter tanggal: 2015-01-01 s/d 2015-12-31
Output file: detik2015.csv
Debug: ON



▶️  Lanjutkan scraping? (y/n):  y



🚀 Mulai scraping...


🔍 Keywords: BBCA, BCA, Bank Central Asia
📄 Mode halaman: SEMUA
📅 Rentang tanggal: 2015-01-01 s/d 2015-12-31
💾 Output: detik2015.csv
🐞 Debug mode aktif


[1/3] 🔑 Keyword: 'BBCA'
----------------------------------------------------------------------
   📄 Halaman 1
   📅 Filter tanggal: 01/01/2015 - 31/12/2015
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=1&sortby=time&fromdatex=01/01/2015&todatex=31/12/2015
   🐞 Debug HTML: debug_html\search_BBCA_page1.html
   🎯 Ditemukan 11 artikel
   ➕ Artikel baru: 11
   📄 Halaman 2
   📅 Filter tanggal: 01/01/2015 - 31/12/2015
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=2&sortby=time&fromdatex=01/01/2015&todatex=31/12/2015
   🐞 Debug HTML: debug_html\search_BBCA_page2.html
   🎯 Ditemukan 11 artikel
   ➕ Artikel baru: 11
   📄 Halaman 3
   📅 Filter tanggal: 01/01/2015 - 31/12/2015
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=3&sortby=time&fromdatex=01/01/

...

         🐞 Debug HTML: debug_html\ini-daftar-emiten-dengan-predikat-gcg-terbaik-di-2_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         ✅ Konten panjang (800 karakter)


Scraping artikel: 100%|██████████▉| 1131/1132 [55:07<00:02,  2.95s/it]

         🐞 Debug HTML: debug_html\ini-perusahaan-yang-rajin-laporkan-utang-dan-berki_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (2048 karakter)


Scraping artikel: 100%|███████████| 1132/1132 [55:10<00:00,  2.92s/it]


💾 Menyimpan hasil ke detik2015.csv...
✅ File tersimpan!

📊 HASIL AKHIR
Total artikel ditemukan: 1132
Berhasil di-scrape: 1128
Gagal di-scrape: 4
File output: detik2015.csv
✅ SCRAPING SELESAI!



### 2016 Detik Finance

In [3]:
import requests as req
from bs4 import BeautifulSoup as bs
import csv
import datetime
import time
from typing import List, Dict
import os
import re
from tqdm import tqdm
from urllib.parse import quote

# KEYWORDS untuk filter artikel
KEYWORDS = ["BBCA", "Bank Central Asia", "BCA"]

def save_debug_html(html_content: str, filename: str):
    """Simpan HTML untuk debugging"""
    debug_dir = "debug_html"
    if not os.path.exists(debug_dir):
        os.makedirs(debug_dir)
    
    filepath = os.path.join(debug_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return filepath


def scrape_article_content(url: str, headers: dict, debug: bool = False) -> str:
    """Scrape konten artikel dari URL Detik"""
    try:
        time.sleep(1.5)
        res = req.get(url, timeout=25, headers=headers)
        if res.status_code != 200:
            if debug:
                print(f"         ❌ HTTP Status: {res.status_code}")
            return ""
        
        soup = bs(res.text, 'lxml')

        # Debug mode: simpan HTML
        if debug:
            filename = re.sub(r'[^\w\-_]', '_', url.split('/')[-1][:50]) + "_article.html"
            saved_path = save_debug_html(res.text, filename)
            print(f"         🐞 Debug HTML: {saved_path}")
        
        # Daftar kemungkinan container konten
        selectors = [
            ('div', 'detail__body-text'),
            ('div', 'itp_bodycontent'),
            ('div', 'detail-content'),
            ('div', 'itp_bodycontent_wrapper'),
            ('div', 'text_detail'),
            ('div', 'detail_text'),
            ('div', 'text-detail'),
            ('div', 'isi_artikel'),
            ('div', 'detail__body'),
            ('div', 'detail_text')
        ]
        
        content_div = None
        for tag, class_name in selectors:
            content_div = soup.find(tag, class_=class_name)
            if content_div:
                if debug:
                    print(f"         🎯 Konten ditemukan: <{tag} class='{class_name}'>")
                break
        
        # Ambil teks
        paragraphs = []
        exclude_prefixes = ['Baca juga', 'Simak', 'ADVERTISEMENT', 'Lihat juga', 'Saksikan']
        
        if content_div:
            for p in content_div.find_all('p'):
                text = p.get_text(strip=True)
                if not text:
                    continue
                if any(text.startswith(prefix) for prefix in exclude_prefixes):
                    continue
                paragraphs.append(text)

        # Fallback jika paragraf kosong
        if not paragraphs:
            if debug:
                print("         ⚠️  Fallback: ambil semua <p> di halaman...")
            for p in soup.find_all('p'):
                text = p.get_text(strip=True)
                if len(text) > 30 and not any(text.startswith(prefix) for prefix in exclude_prefixes):
                    paragraphs.append(text)
        
        # Gabung hasil
        content = ' '.join(paragraphs).strip()

        # Jika masih kosong, coba semua <div> berisi kalimat panjang
        if not content or len(content) < 100:
            long_divs = [div.get_text(strip=True) for div in soup.find_all('div') if len(div.get_text(strip=True)) > 100]
            if long_divs:
                content = ' '.join(long_divs[:3])
                if debug:
                    print("         🧩 Mengambil konten alternatif dari <div> panjang")

        # Simpan meskipun pendek
        if len(content) < 100:
            if debug:
                print(f"         ⚠️  Konten pendek ({len(content)} karakter) — tetap disimpan.")
        else:
            if debug:
                print(f"         ✅ Konten panjang ({len(content)} karakter)")

        return content

    except req.exceptions.Timeout:
        if debug:
            print("         ⚠️  Timeout saat mengakses artikel")
        return ""
    except Exception as e:
        if debug:
            print(f"         ⚠️  Error ambil konten: {type(e).__name__} - {e}")
        return ""


def extract_date_from_text(date_text: str) -> str:
    """Ekstrak dan format tanggal dari teks Detik"""
    try:
        if ',' in date_text:
            date_part = date_text.split(',')[1].strip()
            date_only = ' '.join(date_part.split()[:3])
            months = {
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
                'Mei': '05', 'Jun': '06', 'Jul': '07', 'Agu': '08',
                'Sep': '09', 'Okt': '10', 'Nov': '11', 'Des': '12',
                'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
                'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
                'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
            }
            parts = date_only.split()
            if len(parts) == 3:
                day, month, year = parts
                month_num = months.get(month, month)
                return f"{year}-{month_num}-{day.zfill(2)}"
    except:
        pass
    return date_text


def scrape_search_results(keyword: str, page: int, headers: dict, debug: bool = False,
                         start_date: str = None, end_date: str = None) -> List[Dict]:
    """Scrape hasil pencarian dari Detik.com"""
    articles = []
    search_url = f"https://www.detik.com/search/searchall?query={quote(keyword)}&page={page}&sortby=time"
    
    if start_date and end_date:
        try:
            start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
            end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d")
            fromdatex = start_dt.strftime("%d/%m/%Y")
            todatex = end_dt.strftime("%d/%m/%Y")
            search_url += f"&fromdatex={fromdatex}&todatex={todatex}"
            if debug:
                print(f"   📅 Filter tanggal: {fromdatex} - {todatex}")
        except:
            pass
    
    try:
        print(f"   🔍 Mengakses: {search_url}")
        time.sleep(2)
        res = req.get(search_url, timeout=25, headers=headers)
        
        if res.status_code != 200:
            print(f"   ❌ HTTP {res.status_code}")
            return articles
        
        soup = bs(res.text, 'lxml')
        if debug:
            safe_keyword = re.sub(r'[^\w\-_]', '_', keyword)
            debug_path = save_debug_html(res.text, f"search_{safe_keyword}_page{page}.html")
            print(f"   🐞 Debug HTML: {debug_path}")
        
        article_items = soup.find_all('article') or soup.find_all('div', class_='list-content__item')
        
        if debug:
            print(f"   🎯 Ditemukan {len(article_items)} artikel")
        if not article_items:
            print(f"   ⚠️  Tidak ada artikel ditemukan di halaman ini")
            return articles
        
        for item in article_items:
            try:
                title = None
                link = None
                released = ""
                title_tag = item.find('h3', class_='media__title') or \
                            item.find('h2', class_='media__title') or \
                            item.find('a', class_='media__link')
                if title_tag:
                    if title_tag.name == 'a':
                        title = title_tag.text.strip()
                        link = title_tag.get('href')
                    else:
                        a_tag = title_tag.find('a')
                        if a_tag:
                            title = a_tag.text.strip()
                            link = a_tag.get('href')
                
                date_tag = item.find('div', class_='media__date') or \
                           item.find('span', class_='media__date')
                if date_tag:
                    released = extract_date_from_text(date_tag.text.strip())
                
                if not title or not link:
                    continue
                
                if not link.startswith('http'):
                    link = 'https://www.detik.com' + link
                
                if 'detik.com' not in link:
                    continue
                
                articles.append({'title': title, 'released': released, 'url': link})
            except Exception as e:
                if debug:
                    print(f"   ⚠️  Error parsing item: {type(e).__name__} - {e}")
                continue
        
        return articles
    except req.exceptions.Timeout:
        print(f"   ❌ Timeout saat mengakses halaman pencarian")
        return articles
    except Exception as e:
        print(f"   ❌ Error scraping halaman: {type(e).__name__} - {e}")
        return articles


# Bagian utama tetap sama (tidak diubah)
# Jadi kamu bisa lanjut dari fungsi `sc_detik_search()` di bawah
# salin kode kamu mulai dari def sc_detik_search(...) sampai akhir



def sc_detik_search(keywords: List[str],
                    max_pages: int = None,
                    output_file: str = 'ress_detik.csv',
                    debug: bool = False,
                    start_date: str = None,
                    end_date: str = None):
    """
    Scraping Detik Finance berdasarkan keyword.
    Jika max_pages=None, maka scraping akan berjalan sampai tidak ada artikel baru.
    """
    
    print(f"\n{'='*70}")
    print(f"🔍 Keywords: {', '.join(keywords)}")
    print(f"📄 Mode halaman: {'SEMUA' if max_pages is None else max_pages}")
    if start_date and end_date:
        print(f"📅 Rentang tanggal: {start_date} s/d {end_date}")
    print(f"💾 Output: {output_file}")
    if debug:
        print(f"🐞 Debug mode aktif")
    print(f"{'='*70}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
    }

    all_articles = []
    seen_urls = set()

    # === SCRAPE SEARCH RESULTS ===
    for kw_idx, keyword in enumerate(keywords, 1):
        print(f"\n[{kw_idx}/{len(keywords)}] 🔑 Keyword: '{keyword}'")
        print(f"{'-'*70}")

        page = 1
        keyword_articles = []

        while True:
            print(f"   📄 Halaman {page}")
            articles = scrape_search_results(keyword, page, headers, debug=debug,
                                             start_date=start_date, end_date=end_date)

            if not articles:
                print(f"   ⚠️  Tidak ada artikel ditemukan, berhenti.")
                break

            new_articles = [a for a in articles if a['url'] not in seen_urls]
            for art in new_articles:
                seen_urls.add(art['url'])
            keyword_articles.extend(new_articles)

            print(f"   ➕ Artikel baru: {len(new_articles)}")

            # Hentikan kondisi
            if not new_articles or len(articles) < 5:
                break
            if max_pages is not None and page >= max_pages:
                print(f"   ⛔ Batas halaman {max_pages} tercapai.")
                break

            page += 1
            time.sleep(2)

        print(f"✅ Total artikel keyword '{keyword}': {len(keyword_articles)}")
        all_articles.extend(keyword_articles)
        time.sleep(3)

    if not all_articles:
        print("\n❌ Tidak ada artikel yang ditemukan untuk semua keyword.")
        return

    # === SCRAPE CONTENT ===
    print(f"\n{'='*70}")
    print("📥 MENGAMBIL KONTEN ARTIKEL")
    print(f"{'='*70}\n")

    scraped_data = []
    success_count = 0
    failed_count = 0

    for art in tqdm(all_articles, desc="Scraping artikel", ncols=70):
        content = scrape_article_content(art['url'], headers, debug=debug)
        if not content or len(content) < 100:
            failed_count += 1
            continue

        scraped_data.append({
            'title': art['title'],
            'released': art['released'],
            'url': art['url'],
            'content': content
        })
        success_count += 1
        time.sleep(1)

    # === SAVE TO CSV ===
    print(f"\n{'='*70}")
    print(f"💾 Menyimpan hasil ke {output_file}...")
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['title', 'released', 'url', 'content'], quoting=csv.QUOTE_ALL)
            writer.writeheader()
            writer.writerows(scraped_data)
        print("✅ File tersimpan!\n")
    except Exception as e:
        print(f"❌ Error saat menyimpan file: {e}")

    # === SUMMARY ===
    print(f"{'='*70}")
    print("📊 HASIL AKHIR")
    print(f"{'='*70}")
    print(f"Total artikel ditemukan: {len(all_articles)}")
    print(f"Berhasil di-scrape: {success_count}")
    print(f"Gagal di-scrape: {failed_count}")
    print(f"File output: {output_file}")
    print(f"{'='*70}")
    print("✅ SCRAPING SELESAI!\n")


# === MAIN ===
if __name__ == '__main__':
    print("="*70)
    print("📰 DETIK FINANCE SCRAPER")
    print("="*70)

    default_keywords = ["BBCA", "BCA", "Bank Central Asia"]
    print(f"\n🔍 Keywords default: {', '.join(default_keywords)}")
    use_default = input("Gunakan default keywords? (y/n): ").strip().lower() != 'n'
    keywords = default_keywords if use_default else [
        k.strip() for k in input("Masukkan keyword (pisahkan dengan koma): ").split(',') if k.strip()
    ]

    max_pages_input = input("\n📄 Max halaman per keyword (Enter = semua): ").strip()
    max_pages = int(max_pages_input) if max_pages_input.isdigit() else None

    date_filter = input("\n📅 Aktifkan filter tanggal? (y/n): ").strip().lower() == 'y'
    start_date = end_date = None
    if date_filter:
        start_date = input("   Tanggal MULAI (YYYY-MM-DD): ").strip()
        end_date = input("   Tanggal SELESAI (YYYY-MM-DD): ").strip()

    output_file = input("\n💾 Nama file output (Enter = ress_detik.csv): ").strip() or 'ress_detik.csv'
    debug = input("\n🐞 Aktifkan DEBUG MODE? (y/n): ").strip().lower() == 'y'

    print(f"\n{'='*70}")
    print("📋 KONFIRMASI")
    print(f"{'='*70}")
    print(f"Keywords: {', '.join(keywords)}")
    print(f"Max halaman: {max_pages or 'SEMUA'}")
    if date_filter:
        print(f"Filter tanggal: {start_date} s/d {end_date}")
    print(f"Output file: {output_file}")
    print(f"Debug: {'ON' if debug else 'OFF'}")
    print(f"{'='*70}")

    if input("\n▶️  Lanjutkan scraping? (y/n): ").strip().lower() == 'y':
        print("\n🚀 Mulai scraping...\n")
        sc_detik_search(keywords, max_pages, output_file, debug, start_date, end_date)
    else:
        print("\n❌ Dibatalkan oleh pengguna.")

📰 DETIK FINANCE SCRAPER

🔍 Keywords default: BBCA, BCA, Bank Central Asia


Gunakan default keywords? (y/n):  y

📄 Max halaman per keyword (Enter = semua):  100

📅 Aktifkan filter tanggal? (y/n):  y
   Tanggal MULAI (YYYY-MM-DD):  2016-01-01
   Tanggal SELESAI (YYYY-MM-DD):  2016-12-31

💾 Nama file output (Enter = ress_detik.csv):  detik2016.csv

🐞 Aktifkan DEBUG MODE? (y/n):  y



📋 KONFIRMASI
Keywords: BBCA, BCA, Bank Central Asia
Max halaman: 100
Filter tanggal: 2016-01-01 s/d 2016-12-31
Output file: detik2016.csv
Debug: ON



▶️  Lanjutkan scraping? (y/n):  y



🚀 Mulai scraping...


🔍 Keywords: BBCA, BCA, Bank Central Asia
📄 Mode halaman: 100
📅 Rentang tanggal: 2016-01-01 s/d 2016-12-31
💾 Output: detik2016.csv
🐞 Debug mode aktif


[1/3] 🔑 Keyword: 'BBCA'
----------------------------------------------------------------------
   📄 Halaman 1
   📅 Filter tanggal: 01/01/2016 - 31/12/2016
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=1&sortby=time&fromdatex=01/01/2016&todatex=31/12/2016
   🐞 Debug HTML: debug_html\search_BBCA_page1.html
   🎯 Ditemukan 10 artikel
   ➕ Artikel baru: 10
   📄 Halaman 2
   📅 Filter tanggal: 01/01/2016 - 31/12/2016
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=2&sortby=time&fromdatex=01/01/2016&todatex=31/12/2016
   🐞 Debug HTML: debug_html\search_BBCA_page2.html
   🎯 Ditemukan 10 artikel
   ➕ Artikel baru: 10
   📄 Halaman 3
   📅 Filter tanggal: 01/01/2016 - 31/12/2016
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=3&sortby=time&fromdatex=01/01/20

...

         🐞 Debug HTML: debug_html\ini-jadwal-kegiatan-operasional-bi-di-libur-natal-_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (898 karakter)


Scraping artikel: 100%|████████████▉| 765/766 [37:30<00:02,  2.91s/it]

         🐞 Debug HTML: debug_html\mau-investasi-bunga-6-6-per-tahun-beli-ori-013_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (2048 karakter)


Scraping artikel: 100%|█████████████| 766/766 [37:33<00:00,  2.94s/it]


💾 Menyimpan hasil ke detik2016.csv...
✅ File tersimpan!

📊 HASIL AKHIR
Total artikel ditemukan: 766
Berhasil di-scrape: 765
Gagal di-scrape: 1
File output: detik2016.csv
✅ SCRAPING SELESAI!



### 2017 Detik Finance

In [4]:
import requests as req
from bs4 import BeautifulSoup as bs
import csv
import datetime
import time
from typing import List, Dict
import os
import re
from tqdm import tqdm
from urllib.parse import quote

# KEYWORDS untuk filter artikel
KEYWORDS = ["BBCA", "Bank Central Asia", "BCA"]

def save_debug_html(html_content: str, filename: str):
    """Simpan HTML untuk debugging"""
    debug_dir = "debug_html"
    if not os.path.exists(debug_dir):
        os.makedirs(debug_dir)
    
    filepath = os.path.join(debug_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return filepath


def scrape_article_content(url: str, headers: dict, debug: bool = False) -> str:
    """Scrape konten artikel dari URL Detik"""
    try:
        time.sleep(1.5)
        res = req.get(url, timeout=25, headers=headers)
        if res.status_code != 200:
            if debug:
                print(f"         ❌ HTTP Status: {res.status_code}")
            return ""
        
        soup = bs(res.text, 'lxml')

        # Debug mode: simpan HTML
        if debug:
            filename = re.sub(r'[^\w\-_]', '_', url.split('/')[-1][:50]) + "_article.html"
            saved_path = save_debug_html(res.text, filename)
            print(f"         🐞 Debug HTML: {saved_path}")
        
        # Daftar kemungkinan container konten
        selectors = [
            ('div', 'detail__body-text'),
            ('div', 'itp_bodycontent'),
            ('div', 'detail-content'),
            ('div', 'itp_bodycontent_wrapper'),
            ('div', 'text_detail'),
            ('div', 'detail_text'),
            ('div', 'text-detail'),
            ('div', 'isi_artikel'),
            ('div', 'detail__body'),
            ('div', 'detail_text')
        ]
        
        content_div = None
        for tag, class_name in selectors:
            content_div = soup.find(tag, class_=class_name)
            if content_div:
                if debug:
                    print(f"         🎯 Konten ditemukan: <{tag} class='{class_name}'>")
                break
        
        # Ambil teks
        paragraphs = []
        exclude_prefixes = ['Baca juga', 'Simak', 'ADVERTISEMENT', 'Lihat juga', 'Saksikan']
        
        if content_div:
            for p in content_div.find_all('p'):
                text = p.get_text(strip=True)
                if not text:
                    continue
                if any(text.startswith(prefix) for prefix in exclude_prefixes):
                    continue
                paragraphs.append(text)

        # Fallback jika paragraf kosong
        if not paragraphs:
            if debug:
                print("         ⚠️  Fallback: ambil semua <p> di halaman...")
            for p in soup.find_all('p'):
                text = p.get_text(strip=True)
                if len(text) > 30 and not any(text.startswith(prefix) for prefix in exclude_prefixes):
                    paragraphs.append(text)
        
        # Gabung hasil
        content = ' '.join(paragraphs).strip()

        # Jika masih kosong, coba semua <div> berisi kalimat panjang
        if not content or len(content) < 100:
            long_divs = [div.get_text(strip=True) for div in soup.find_all('div') if len(div.get_text(strip=True)) > 100]
            if long_divs:
                content = ' '.join(long_divs[:3])
                if debug:
                    print("         🧩 Mengambil konten alternatif dari <div> panjang")

        # Simpan meskipun pendek
        if len(content) < 100:
            if debug:
                print(f"         ⚠️  Konten pendek ({len(content)} karakter) — tetap disimpan.")
        else:
            if debug:
                print(f"         ✅ Konten panjang ({len(content)} karakter)")

        return content

    except req.exceptions.Timeout:
        if debug:
            print("         ⚠️  Timeout saat mengakses artikel")
        return ""
    except Exception as e:
        if debug:
            print(f"         ⚠️  Error ambil konten: {type(e).__name__} - {e}")
        return ""


def extract_date_from_text(date_text: str) -> str:
    """Ekstrak dan format tanggal dari teks Detik"""
    try:
        if ',' in date_text:
            date_part = date_text.split(',')[1].strip()
            date_only = ' '.join(date_part.split()[:3])
            months = {
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
                'Mei': '05', 'Jun': '06', 'Jul': '07', 'Agu': '08',
                'Sep': '09', 'Okt': '10', 'Nov': '11', 'Des': '12',
                'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
                'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
                'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
            }
            parts = date_only.split()
            if len(parts) == 3:
                day, month, year = parts
                month_num = months.get(month, month)
                return f"{year}-{month_num}-{day.zfill(2)}"
    except:
        pass
    return date_text


def scrape_search_results(keyword: str, page: int, headers: dict, debug: bool = False,
                         start_date: str = None, end_date: str = None) -> List[Dict]:
    """Scrape hasil pencarian dari Detik.com"""
    articles = []
    search_url = f"https://www.detik.com/search/searchall?query={quote(keyword)}&page={page}&sortby=time"
    
    if start_date and end_date:
        try:
            start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
            end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d")
            fromdatex = start_dt.strftime("%d/%m/%Y")
            todatex = end_dt.strftime("%d/%m/%Y")
            search_url += f"&fromdatex={fromdatex}&todatex={todatex}"
            if debug:
                print(f"   📅 Filter tanggal: {fromdatex} - {todatex}")
        except:
            pass
    
    try:
        print(f"   🔍 Mengakses: {search_url}")
        time.sleep(2)
        res = req.get(search_url, timeout=25, headers=headers)
        
        if res.status_code != 200:
            print(f"   ❌ HTTP {res.status_code}")
            return articles
        
        soup = bs(res.text, 'lxml')
        if debug:
            safe_keyword = re.sub(r'[^\w\-_]', '_', keyword)
            debug_path = save_debug_html(res.text, f"search_{safe_keyword}_page{page}.html")
            print(f"   🐞 Debug HTML: {debug_path}")
        
        article_items = soup.find_all('article') or soup.find_all('div', class_='list-content__item')
        
        if debug:
            print(f"   🎯 Ditemukan {len(article_items)} artikel")
        if not article_items:
            print(f"   ⚠️  Tidak ada artikel ditemukan di halaman ini")
            return articles
        
        for item in article_items:
            try:
                title = None
                link = None
                released = ""
                title_tag = item.find('h3', class_='media__title') or \
                            item.find('h2', class_='media__title') or \
                            item.find('a', class_='media__link')
                if title_tag:
                    if title_tag.name == 'a':
                        title = title_tag.text.strip()
                        link = title_tag.get('href')
                    else:
                        a_tag = title_tag.find('a')
                        if a_tag:
                            title = a_tag.text.strip()
                            link = a_tag.get('href')
                
                date_tag = item.find('div', class_='media__date') or \
                           item.find('span', class_='media__date')
                if date_tag:
                    released = extract_date_from_text(date_tag.text.strip())
                
                if not title or not link:
                    continue
                
                if not link.startswith('http'):
                    link = 'https://www.detik.com' + link
                
                if 'detik.com' not in link:
                    continue
                
                articles.append({'title': title, 'released': released, 'url': link})
            except Exception as e:
                if debug:
                    print(f"   ⚠️  Error parsing item: {type(e).__name__} - {e}")
                continue
        
        return articles
    except req.exceptions.Timeout:
        print(f"   ❌ Timeout saat mengakses halaman pencarian")
        return articles
    except Exception as e:
        print(f"   ❌ Error scraping halaman: {type(e).__name__} - {e}")
        return articles


# Bagian utama tetap sama (tidak diubah)
# Jadi kamu bisa lanjut dari fungsi `sc_detik_search()` di bawah
# salin kode kamu mulai dari def sc_detik_search(...) sampai akhir



def sc_detik_search(keywords: List[str],
                    max_pages: int = None,
                    output_file: str = 'ress_detik.csv',
                    debug: bool = False,
                    start_date: str = None,
                    end_date: str = None):
    """
    Scraping Detik Finance berdasarkan keyword.
    Jika max_pages=None, maka scraping akan berjalan sampai tidak ada artikel baru.
    """
    
    print(f"\n{'='*70}")
    print(f"🔍 Keywords: {', '.join(keywords)}")
    print(f"📄 Mode halaman: {'SEMUA' if max_pages is None else max_pages}")
    if start_date and end_date:
        print(f"📅 Rentang tanggal: {start_date} s/d {end_date}")
    print(f"💾 Output: {output_file}")
    if debug:
        print(f"🐞 Debug mode aktif")
    print(f"{'='*70}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
    }

    all_articles = []
    seen_urls = set()

    # === SCRAPE SEARCH RESULTS ===
    for kw_idx, keyword in enumerate(keywords, 1):
        print(f"\n[{kw_idx}/{len(keywords)}] 🔑 Keyword: '{keyword}'")
        print(f"{'-'*70}")

        page = 1
        keyword_articles = []

        while True:
            print(f"   📄 Halaman {page}")
            articles = scrape_search_results(keyword, page, headers, debug=debug,
                                             start_date=start_date, end_date=end_date)

            if not articles:
                print(f"   ⚠️  Tidak ada artikel ditemukan, berhenti.")
                break

            new_articles = [a for a in articles if a['url'] not in seen_urls]
            for art in new_articles:
                seen_urls.add(art['url'])
            keyword_articles.extend(new_articles)

            print(f"   ➕ Artikel baru: {len(new_articles)}")

            # Hentikan kondisi
            if not new_articles or len(articles) < 5:
                break
            if max_pages is not None and page >= max_pages:
                print(f"   ⛔ Batas halaman {max_pages} tercapai.")
                break

            page += 1
            time.sleep(2)

        print(f"✅ Total artikel keyword '{keyword}': {len(keyword_articles)}")
        all_articles.extend(keyword_articles)
        time.sleep(3)

    if not all_articles:
        print("\n❌ Tidak ada artikel yang ditemukan untuk semua keyword.")
        return

    # === SCRAPE CONTENT ===
    print(f"\n{'='*70}")
    print("📥 MENGAMBIL KONTEN ARTIKEL")
    print(f"{'='*70}\n")

    scraped_data = []
    success_count = 0
    failed_count = 0

    for art in tqdm(all_articles, desc="Scraping artikel", ncols=70):
        content = scrape_article_content(art['url'], headers, debug=debug)
        if not content or len(content) < 100:
            failed_count += 1
            continue

        scraped_data.append({
            'title': art['title'],
            'released': art['released'],
            'url': art['url'],
            'content': content
        })
        success_count += 1
        time.sleep(1)

    # === SAVE TO CSV ===
    print(f"\n{'='*70}")
    print(f"💾 Menyimpan hasil ke {output_file}...")
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['title', 'released', 'url', 'content'], quoting=csv.QUOTE_ALL)
            writer.writeheader()
            writer.writerows(scraped_data)
        print("✅ File tersimpan!\n")
    except Exception as e:
        print(f"❌ Error saat menyimpan file: {e}")

    # === SUMMARY ===
    print(f"{'='*70}")
    print("📊 HASIL AKHIR")
    print(f"{'='*70}")
    print(f"Total artikel ditemukan: {len(all_articles)}")
    print(f"Berhasil di-scrape: {success_count}")
    print(f"Gagal di-scrape: {failed_count}")
    print(f"File output: {output_file}")
    print(f"{'='*70}")
    print("✅ SCRAPING SELESAI!\n")


# === MAIN ===
if __name__ == '__main__':
    print("="*70)
    print("📰 DETIK FINANCE SCRAPER")
    print("="*70)

    default_keywords = ["BBCA", "BCA", "Bank Central Asia"]
    print(f"\n🔍 Keywords default: {', '.join(default_keywords)}")
    use_default = input("Gunakan default keywords? (y/n): ").strip().lower() != 'n'
    keywords = default_keywords if use_default else [
        k.strip() for k in input("Masukkan keyword (pisahkan dengan koma): ").split(',') if k.strip()
    ]

    max_pages_input = input("\n📄 Max halaman per keyword (Enter = semua): ").strip()
    max_pages = int(max_pages_input) if max_pages_input.isdigit() else None

    date_filter = input("\n📅 Aktifkan filter tanggal? (y/n): ").strip().lower() == 'y'
    start_date = end_date = None
    if date_filter:
        start_date = input("   Tanggal MULAI (YYYY-MM-DD): ").strip()
        end_date = input("   Tanggal SELESAI (YYYY-MM-DD): ").strip()

    output_file = input("\n💾 Nama file output (Enter = ress_detik.csv): ").strip() or 'ress_detik.csv'
    debug = input("\n🐞 Aktifkan DEBUG MODE? (y/n): ").strip().lower() == 'y'

    print(f"\n{'='*70}")
    print("📋 KONFIRMASI")
    print(f"{'='*70}")
    print(f"Keywords: {', '.join(keywords)}")
    print(f"Max halaman: {max_pages or 'SEMUA'}")
    if date_filter:
        print(f"Filter tanggal: {start_date} s/d {end_date}")
    print(f"Output file: {output_file}")
    print(f"Debug: {'ON' if debug else 'OFF'}")
    print(f"{'='*70}")

    if input("\n▶️  Lanjutkan scraping? (y/n): ").strip().lower() == 'y':
        print("\n🚀 Mulai scraping...\n")
        sc_detik_search(keywords, max_pages, output_file, debug, start_date, end_date)
    else:
        print("\n❌ Dibatalkan oleh pengguna.")

📰 DETIK FINANCE SCRAPER

🔍 Keywords default: BBCA, BCA, Bank Central Asia


Gunakan default keywords? (y/n):  y

📄 Max halaman per keyword (Enter = semua):  semua

📅 Aktifkan filter tanggal? (y/n):  y
   Tanggal MULAI (YYYY-MM-DD):  2017-01-01
   Tanggal SELESAI (YYYY-MM-DD):  2017-12-31

💾 Nama file output (Enter = ress_detik.csv):  detik2017.csv

🐞 Aktifkan DEBUG MODE? (y/n):  y



📋 KONFIRMASI
Keywords: BBCA, BCA, Bank Central Asia
Max halaman: SEMUA
Filter tanggal: 2017-01-01 s/d 2017-12-31
Output file: detik2017.csv
Debug: ON



▶️  Lanjutkan scraping? (y/n):  y



🚀 Mulai scraping...


🔍 Keywords: BBCA, BCA, Bank Central Asia
📄 Mode halaman: SEMUA
📅 Rentang tanggal: 2017-01-01 s/d 2017-12-31
💾 Output: detik2017.csv
🐞 Debug mode aktif


[1/3] 🔑 Keyword: 'BBCA'
----------------------------------------------------------------------
   📄 Halaman 1
   📅 Filter tanggal: 01/01/2017 - 31/12/2017
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=1&sortby=time&fromdatex=01/01/2017&todatex=31/12/2017
   🐞 Debug HTML: debug_html\search_BBCA_page1.html
   🎯 Ditemukan 10 artikel
   ➕ Artikel baru: 10
   📄 Halaman 2
   📅 Filter tanggal: 01/01/2017 - 31/12/2017
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=2&sortby=time&fromdatex=01/01/2017&todatex=31/12/2017
   🐞 Debug HTML: debug_html\search_BBCA_page2.html
   🎯 Ditemukan 10 artikel
   ➕ Artikel baru: 10
   📄 Halaman 3
   📅 Filter tanggal: 01/01/2017 - 31/12/2017
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=3&sortby=time&fromdatex=01/01/

...

         🐞 Debug HTML: debug_html\kesepakatan-ri-malaysia-thailand-tak-pakai-dolar-a_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (886 karakter)


Scraping artikel: 100%|████████████▉| 781/782 [37:58<00:02,  2.88s/it]

         🐞 Debug HTML: debug_html\selama-mudik-pembayaran-non-tunai-di-tol-naik-26_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (2048 karakter)


Scraping artikel: 100%|█████████████| 782/782 [38:01<00:00,  2.92s/it]


💾 Menyimpan hasil ke detik2017.csv...
✅ File tersimpan!

📊 HASIL AKHIR
Total artikel ditemukan: 782
Berhasil di-scrape: 774
Gagal di-scrape: 8
File output: detik2017.csv
✅ SCRAPING SELESAI!



### 2018 Detik Finance

In [8]:
import requests as req
from bs4 import BeautifulSoup as bs
import csv
import datetime
import time
from typing import List, Dict
import os
import re
from tqdm import tqdm
from urllib.parse import quote

# KEYWORDS untuk filter artikel
KEYWORDS = ["BBCA", "Bank Central Asia", "BCA"]

def save_debug_html(html_content: str, filename: str):
    """Simpan HTML untuk debugging"""
    debug_dir = "debug_html"
    if not os.path.exists(debug_dir):
        os.makedirs(debug_dir)
    
    filepath = os.path.join(debug_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    return filepath


def scrape_article_content(url: str, headers: dict, debug: bool = False) -> str:
    """Scrape konten artikel dari URL Detik"""
    try:
        time.sleep(1.5)
        res = req.get(url, timeout=25, headers=headers)
        if res.status_code != 200:
            if debug:
                print(f"         ❌ HTTP Status: {res.status_code}")
            return ""
        
        soup = bs(res.text, 'lxml')

        # Debug mode: simpan HTML
        if debug:
            filename = re.sub(r'[^\w\-_]', '_', url.split('/')[-1][:50]) + "_article.html"
            saved_path = save_debug_html(res.text, filename)
            print(f"         🐞 Debug HTML: {saved_path}")
        
        # Daftar kemungkinan container konten
        selectors = [
            ('div', 'detail__body-text'),
            ('div', 'itp_bodycontent'),
            ('div', 'detail-content'),
            ('div', 'itp_bodycontent_wrapper'),
            ('div', 'text_detail'),
            ('div', 'detail_text'),
            ('div', 'text-detail'),
            ('div', 'isi_artikel'),
            ('div', 'detail__body'),
            ('div', 'detail_text')
        ]
        
        content_div = None
        for tag, class_name in selectors:
            content_div = soup.find(tag, class_=class_name)
            if content_div:
                if debug:
                    print(f"         🎯 Konten ditemukan: <{tag} class='{class_name}'>")
                break
        
        # Ambil teks
        paragraphs = []
        exclude_prefixes = ['Baca juga', 'Simak', 'ADVERTISEMENT', 'Lihat juga', 'Saksikan']
        
        if content_div:
            for p in content_div.find_all('p'):
                text = p.get_text(strip=True)
                if not text:
                    continue
                if any(text.startswith(prefix) for prefix in exclude_prefixes):
                    continue
                paragraphs.append(text)

        # Fallback jika paragraf kosong
        if not paragraphs:
            if debug:
                print("         ⚠️  Fallback: ambil semua <p> di halaman...")
            for p in soup.find_all('p'):
                text = p.get_text(strip=True)
                if len(text) > 30 and not any(text.startswith(prefix) for prefix in exclude_prefixes):
                    paragraphs.append(text)
        
        # Gabung hasil
        content = ' '.join(paragraphs).strip()

        # Jika masih kosong, coba semua <div> berisi kalimat panjang
        if not content or len(content) < 100:
            long_divs = [div.get_text(strip=True) for div in soup.find_all('div') if len(div.get_text(strip=True)) > 100]
            if long_divs:
                content = ' '.join(long_divs[:3])
                if debug:
                    print("         🧩 Mengambil konten alternatif dari <div> panjang")

        # Simpan meskipun pendek
        if len(content) < 100:
            if debug:
                print(f"         ⚠️  Konten pendek ({len(content)} karakter) — tetap disimpan.")
        else:
            if debug:
                print(f"         ✅ Konten panjang ({len(content)} karakter)")

        return content

    except req.exceptions.Timeout:
        if debug:
            print("         ⚠️  Timeout saat mengakses artikel")
        return ""
    except Exception as e:
        if debug:
            print(f"         ⚠️  Error ambil konten: {type(e).__name__} - {e}")
        return ""


def extract_date_from_text(date_text: str) -> str:
    """Ekstrak dan format tanggal dari teks Detik"""
    try:
        if ',' in date_text:
            date_part = date_text.split(',')[1].strip()
            date_only = ' '.join(date_part.split()[:3])
            months = {
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
                'Mei': '05', 'Jun': '06', 'Jul': '07', 'Agu': '08',
                'Sep': '09', 'Okt': '10', 'Nov': '11', 'Des': '12',
                'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
                'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
                'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
            }
            parts = date_only.split()
            if len(parts) == 3:
                day, month, year = parts
                month_num = months.get(month, month)
                return f"{year}-{month_num}-{day.zfill(2)}"
    except:
        pass
    return date_text


def scrape_search_results(keyword: str, page: int, headers: dict, debug: bool = False,
                         start_date: str = None, end_date: str = None) -> List[Dict]:
    """Scrape hasil pencarian dari Detik.com"""
    articles = []
    search_url = f"https://www.detik.com/search/searchall?query={quote(keyword)}&page={page}&sortby=time"
    
    if start_date and end_date:
        try:
            start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
            end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d")
            fromdatex = start_dt.strftime("%d/%m/%Y")
            todatex = end_dt.strftime("%d/%m/%Y")
            search_url += f"&fromdatex={fromdatex}&todatex={todatex}"
            if debug:
                print(f"   📅 Filter tanggal: {fromdatex} - {todatex}")
        except:
            pass
    
    try:
        print(f"   🔍 Mengakses: {search_url}")
        time.sleep(2)
        res = req.get(search_url, timeout=25, headers=headers)
        
        if res.status_code != 200:
            print(f"   ❌ HTTP {res.status_code}")
            return articles
        
        soup = bs(res.text, 'lxml')
        if debug:
            safe_keyword = re.sub(r'[^\w\-_]', '_', keyword)
            debug_path = save_debug_html(res.text, f"search_{safe_keyword}_page{page}.html")
            print(f"   🐞 Debug HTML: {debug_path}")
        
        article_items = soup.find_all('article') or soup.find_all('div', class_='list-content__item')
        
        if debug:
            print(f"   🎯 Ditemukan {len(article_items)} artikel")
        if not article_items:
            print(f"   ⚠️  Tidak ada artikel ditemukan di halaman ini")
            return articles
        
        for item in article_items:
            try:
                title = None
                link = None
                released = ""
                title_tag = item.find('h3', class_='media__title') or \
                            item.find('h2', class_='media__title') or \
                            item.find('a', class_='media__link')
                if title_tag:
                    if title_tag.name == 'a':
                        title = title_tag.text.strip()
                        link = title_tag.get('href')
                    else:
                        a_tag = title_tag.find('a')
                        if a_tag:
                            title = a_tag.text.strip()
                            link = a_tag.get('href')
                
                date_tag = item.find('div', class_='media__date') or \
                           item.find('span', class_='media__date')
                if date_tag:
                    released = extract_date_from_text(date_tag.text.strip())
                
                if not title or not link:
                    continue
                
                if not link.startswith('http'):
                    link = 'https://www.detik.com' + link
                
                if 'detik.com' not in link:
                    continue
                
                articles.append({'title': title, 'released': released, 'url': link})
            except Exception as e:
                if debug:
                    print(f"   ⚠️  Error parsing item: {type(e).__name__} - {e}")
                continue
        
        return articles
    except req.exceptions.Timeout:
        print(f"   ❌ Timeout saat mengakses halaman pencarian")
        return articles
    except Exception as e:
        print(f"   ❌ Error scraping halaman: {type(e).__name__} - {e}")
        return articles


# Bagian utama tetap sama (tidak diubah)
# Jadi kamu bisa lanjut dari fungsi `sc_detik_search()` di bawah
# salin kode kamu mulai dari def sc_detik_search(...) sampai akhir



def sc_detik_search(keywords: List[str],
                    max_pages: int = None,
                    output_file: str = 'ress_detik.csv',
                    debug: bool = False,
                    start_date: str = None,
                    end_date: str = None):
    """
    Scraping Detik Finance berdasarkan keyword.
    Jika max_pages=None, maka scraping akan berjalan sampai tidak ada artikel baru.
    """
    
    print(f"\n{'='*70}")
    print(f"🔍 Keywords: {', '.join(keywords)}")
    print(f"📄 Mode halaman: {'SEMUA' if max_pages is None else max_pages}")
    if start_date and end_date:
        print(f"📅 Rentang tanggal: {start_date} s/d {end_date}")
    print(f"💾 Output: {output_file}")
    if debug:
        print(f"🐞 Debug mode aktif")
    print(f"{'='*70}\n")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
    }

    all_articles = []
    seen_urls = set()

    # === SCRAPE SEARCH RESULTS ===
    for kw_idx, keyword in enumerate(keywords, 1):
        print(f"\n[{kw_idx}/{len(keywords)}] 🔑 Keyword: '{keyword}'")
        print(f"{'-'*70}")

        page = 1
        keyword_articles = []

        while True:
            print(f"   📄 Halaman {page}")
            articles = scrape_search_results(keyword, page, headers, debug=debug,
                                             start_date=start_date, end_date=end_date)

            if not articles:
                print(f"   ⚠️  Tidak ada artikel ditemukan, berhenti.")
                break

            new_articles = [a for a in articles if a['url'] not in seen_urls]
            for art in new_articles:
                seen_urls.add(art['url'])
            keyword_articles.extend(new_articles)

            print(f"   ➕ Artikel baru: {len(new_articles)}")

            # Hentikan kondisi
            if not new_articles or len(articles) < 5:
                break
            if max_pages is not None and page >= max_pages:
                print(f"   ⛔ Batas halaman {max_pages} tercapai.")
                break

            page += 1
            time.sleep(2)

        print(f"✅ Total artikel keyword '{keyword}': {len(keyword_articles)}")
        all_articles.extend(keyword_articles)
        time.sleep(3)

    if not all_articles:
        print("\n❌ Tidak ada artikel yang ditemukan untuk semua keyword.")
        return

    # === SCRAPE CONTENT ===
    print(f"\n{'='*70}")
    print("📥 MENGAMBIL KONTEN ARTIKEL")
    print(f"{'='*70}\n")

    scraped_data = []
    success_count = 0
    failed_count = 0

    for art in tqdm(all_articles, desc="Scraping artikel", ncols=70):
        content = scrape_article_content(art['url'], headers, debug=debug)
        if not content or len(content) < 100:
            failed_count += 1
            continue

        scraped_data.append({
            'title': art['title'],
            'released': art['released'],
            'url': art['url'],
            'content': content
        })
        success_count += 1
        time.sleep(1)

    # === SAVE TO CSV ===
    print(f"\n{'='*70}")
    print(f"💾 Menyimpan hasil ke {output_file}...")
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['title', 'released', 'url', 'content'], quoting=csv.QUOTE_ALL)
            writer.writeheader()
            writer.writerows(scraped_data)
        print("✅ File tersimpan!\n")
    except Exception as e:
        print(f"❌ Error saat menyimpan file: {e}")

    # === SUMMARY ===
    print(f"{'='*70}")
    print("📊 HASIL AKHIR")
    print(f"{'='*70}")
    print(f"Total artikel ditemukan: {len(all_articles)}")
    print(f"Berhasil di-scrape: {success_count}")
    print(f"Gagal di-scrape: {failed_count}")
    print(f"File output: {output_file}")
    print(f"{'='*70}")
    print("✅ SCRAPING SELESAI!\n")


# === MAIN ===
if __name__ == '__main__':
    print("="*70)
    print("📰 DETIK FINANCE SCRAPER")
    print("="*70)

    default_keywords = ["BBCA", "BCA", "Bank Central Asia"]
    print(f"\n🔍 Keywords default: {', '.join(default_keywords)}")
    use_default = input("Gunakan default keywords? (y/n): ").strip().lower() != 'n'
    keywords = default_keywords if use_default else [
        k.strip() for k in input("Masukkan keyword (pisahkan dengan koma): ").split(',') if k.strip()
    ]

    max_pages_input = input("\n📄 Max halaman per keyword (Enter = semua): ").strip()
    max_pages = int(max_pages_input) if max_pages_input.isdigit() else None

    date_filter = input("\n📅 Aktifkan filter tanggal? (y/n): ").strip().lower() == 'y'
    start_date = end_date = None
    if date_filter:
        start_date = input("   Tanggal MULAI (YYYY-MM-DD): ").strip()
        end_date = input("   Tanggal SELESAI (YYYY-MM-DD): ").strip()

    output_file = input("\n💾 Nama file output (Enter = ress_detik.csv): ").strip() or 'ress_detik.csv'
    debug = input("\n🐞 Aktifkan DEBUG MODE? (y/n): ").strip().lower() == 'y'

    print(f"\n{'='*70}")
    print("📋 KONFIRMASI")
    print(f"{'='*70}")
    print(f"Keywords: {', '.join(keywords)}")
    print(f"Max halaman: {max_pages or 'SEMUA'}")
    if date_filter:
        print(f"Filter tanggal: {start_date} s/d {end_date}")
    print(f"Output file: {output_file}")
    print(f"Debug: {'ON' if debug else 'OFF'}")
    print(f"{'='*70}")

    if input("\n▶️  Lanjutkan scraping? (y/n): ").strip().lower() == 'y':
        print("\n🚀 Mulai scraping...\n")
        sc_detik_search(keywords, max_pages, output_file, debug, start_date, end_date)
    else:
        print("\n❌ Dibatalkan oleh pengguna.")

📰 DETIK FINANCE SCRAPER

🔍 Keywords default: BBCA, BCA, Bank Central Asia


Gunakan default keywords? (y/n):  y

📄 Max halaman per keyword (Enter = semua):  100

📅 Aktifkan filter tanggal? (y/n):  y
   Tanggal MULAI (YYYY-MM-DD):  2018-01-01
   Tanggal SELESAI (YYYY-MM-DD):  2018-12-31

💾 Nama file output (Enter = ress_detik.csv):  detik2018.csv

🐞 Aktifkan DEBUG MODE? (y/n):  y



📋 KONFIRMASI
Keywords: BBCA, BCA, Bank Central Asia
Max halaman: 100
Filter tanggal: 2018-01-01 s/d 2018-12-31
Output file: detik2018.csv
Debug: ON



▶️  Lanjutkan scraping? (y/n):  y



🚀 Mulai scraping...


🔍 Keywords: BBCA, BCA, Bank Central Asia
📄 Mode halaman: 100
📅 Rentang tanggal: 2018-01-01 s/d 2018-12-31
💾 Output: detik2018.csv
🐞 Debug mode aktif


[1/3] 🔑 Keyword: 'BBCA'
----------------------------------------------------------------------
   📄 Halaman 1
   📅 Filter tanggal: 01/01/2018 - 31/12/2018
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=1&sortby=time&fromdatex=01/01/2018&todatex=31/12/2018
   🐞 Debug HTML: debug_html\search_BBCA_page1.html
   🎯 Ditemukan 11 artikel
   ➕ Artikel baru: 11
   📄 Halaman 2
   📅 Filter tanggal: 01/01/2018 - 31/12/2018
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=2&sortby=time&fromdatex=01/01/2018&todatex=31/12/2018
   🐞 Debug HTML: debug_html\search_BBCA_page2.html
   🎯 Ditemukan 10 artikel
   ➕ Artikel baru: 10
   📄 Halaman 3
   📅 Filter tanggal: 01/01/2018 - 31/12/2018
   🔍 Mengakses: https://www.detik.com/search/searchall?query=BBCA&page=3&sortby=time&fromdatex=01/01/20

...

         🐞 Debug HTML: debug_html\hakim-minta-jaksa-kpk-buka-blokir-rekening-pengusa_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (2048 karakter)


Scraping artikel: 100%|████████████▉| 692/693 [33:43<00:02,  2.90s/it]

         🐞 Debug HTML: debug_html\bank-jual-dolar-as-sampai-rp-14-785_article.html
         🎯 Konten ditemukan: <div class='detail__body-text'>
         🧩 Mengambil konten alternatif dari <div> panjang
         ✅ Konten panjang (2048 karakter)


Scraping artikel: 100%|█████████████| 693/693 [33:46<00:00,  2.92s/it]


💾 Menyimpan hasil ke detik2018.csv...
✅ File tersimpan!

📊 HASIL AKHIR
Total artikel ditemukan: 693
Berhasil di-scrape: 687
Gagal di-scrape: 6
File output: detik2018.csv
✅ SCRAPING SELESAI!

